# 5m Explore Fashion Similarity and Clusters

Compare nearest-neighbor retrieval with clustering on a small image sample, without confusing clusters with semantic categories.

**Budget:** 15–25 minutes; a CPU and a few thousand images are enough. No Klique endpoint or token is needed for the default path.

**Before you start:** complete the [JupyterLab introduction](05_Running_Jupyter_Lab.ipynb), prepare Python 3.10+ and the packages in the [support guide](README.md), and select that kernel. Run every cell from a fresh kernel in order. The default is offline: cached real datasets are used when present; otherwise the helper explicitly identifies an 8×8 scikit-learn digits fallback. That fallback is **not MNIST, EMNIST, or Fashion-MNIST**, and its metrics are not comparable to theirs. Downloads are opt-in; read [dataset sources and terms](DATASETS.md) first.

Save your work in the approved persistent workspace. No live Klique launch or restore is established by these local exercises.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
LAB = HERE if (HERE / "image_lab.py").is_file() else HERE / "05_Jupyter"
if not (LAB / "image_lab.py").is_file():
    raise RuntimeError("Start this kernel in the repository root or 05_Jupyter folder.")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from image_lab import load_image_data, train_baseline
SEED = 42
OUTPUT = LAB / "lab_outputs"
OUTPUT.mkdir(exist_ok=True)


In [ ]:
ALLOW_DOWNLOAD = False
data = load_image_data("fashion_mnist", allow_download=ALLOW_DOWNLOAD,
                       cache_dir=OUTPUT / "data", limit_train=2000, limit_test=300, seed=SEED)
print("Actual data:", data.name, data.provenance)
print("Class names:", data.class_names)
if data.train_images.shape[1:] != (28, 28):
    print("Offline digit fallback: this run demonstrates the method, not clothing similarity.")


## Fit representation on training data only

Fashion-MNIST contains 28×28 grayscale clothing images from Zalando Research, with ten category labels. Here PCA compresses normalized pixels, then cosine nearest neighbors compare a held-out query with training images. Labels annotate the result only; they do not fit PCA or the neighbor index. Pixel-space similarity is a simple baseline and can favor backgrounds and pose over useful semantics.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
import pandas as pd
X_train = data.train_images.reshape(len(data.train_images), -1)
X_test = data.test_images.reshape(len(data.test_images), -1)
pca = PCA(n_components=24, random_state=SEED)
train_features = pca.fit_transform(X_train)
test_features = pca.transform(X_test)
search = NearestNeighbors(n_neighbors=5, metric="cosine").fit(train_features)
query_index = 0
distances, neighbors = search.kneighbors(test_features[query_index:query_index+1])
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
axes[0].imshow(data.test_images[query_index], cmap="gray"); axes[0].set_title("Held-out query")
for ax, index, distance in zip(axes[1:], neighbors[0], distances[0]):
    ax.imshow(data.train_images[index], cmap="gray")
    ax.set_title(f"{data.class_names[int(data.train_labels[index])]}\n{distance:.2f}")
for ax in axes: ax.axis("off")
plt.show()


## Clusters are not ground-truth labels

K-means uses Euclidean distances in the same PCA space. Cluster IDs are arbitrary and can change between runs. A contingency table relates clusters to known labels for interpretation; it is not classification accuracy. The two-dimensional projection is a lossy visualization, not proof of distinct natural categories.


In [ ]:
clusters = KMeans(n_clusters=10, n_init=5, random_state=SEED).fit_predict(train_features)
contingency = pd.crosstab(pd.Series(clusters, name="cluster"),
    pd.Series([data.class_names[int(i)] for i in data.train_labels], name="known label"))
display(contingency)
plt.scatter(train_features[:, 0], train_features[:, 1], c=clusters, cmap="tab10", s=8)
plt.xlabel("PCA 1"); plt.ylabel("PCA 2"); plt.title(data.name + ": training clusters")
plt.show()
print("Variance retained by PCA:", float(pca.explained_variance_ratio_.sum()))


## Checkpoint, exercise, and cleanup

**Checkpoint:** You can retrieve five training neighbors of a held-out image, inspect cluster composition, and distinguish a clothing run from the honest offline digit fallback.

**Try it:** Choose another held-out query and inspect where visually similar images have different labels. Try a training-only change to the component count and avoid choosing settings from test labels.

**Troubleshooting:** if imports fail, select the kernel where the section requirements were installed. If a data host is blocked, keep downloads off and use the named offline fixture, or ask your administrator to stage the official data in the cache. Do not disable TLS verification. A convergence warning means the solver hit its iteration budget; record it rather than silently claiming an optimum.

Save the notebook and useful artifacts under `lab_outputs/`; it is ignored by Git but may still enter workspace snapshots. Clear outputs before sharing if you used private data. Shut down unused kernels, then stop the managed session and verify its stopped state as in the introduction. Closing a browser tab is not cleanup.

**Next:** [5n: Clean Messy Data](05n_Clean_Messy_Data.ipynb) · **Index:** [Jupyter labs](README.md) · [Orientation](../00_Orientation/00_Orientation.ipynb)
